In [1]:
import os
import json
import duckdb
import hashlib
from datetime import datetime

# Setup paths
FHIR_DIR = "../synthea/output/fhir"
DB_PATH = "../data/omop_clinical.duckdb"

def stable_person_id(source_id: str) -> int:
    """
    Generates a stable, deterministic integer ID from a string.
    Ensures the same patient always gets the exact same ID across different ETL runs.
    """
    return int(hashlib.sha256(source_id.encode()).hexdigest(), 16) % (10**9)

def get_omop_gender_id(gender_string):
    """
    Hardcoded mapping for simple demographic variables.
    Standard OMOP gender concepts: 8507 (Male) and 8532 (Female).
    """
    mapping = {
        'male': 8507,
        'female': 8532
    }
    return mapping.get(gender_string.lower(), 0) # Returns 0 (Unknown) if match fails

def extract_person_data(patient_file):
    """Reads a FHIR bundle and extracts baseline demographic data."""
    file_path = os.path.join(FHIR_DIR, patient_file)
    with open(file_path, 'r', encoding='utf-8') as f:
        fhir_data = json.load(f)
        
    for entry in fhir_data.get('entry', []):
        resource = entry.get('resource', {})
        
        # FHIR stores demographic data in the 'Patient' resource
        if resource.get('resourceType') == 'Patient':
            
            patient_source_id = resource.get('id', 'unknown')
            
            # CRITICAL FIX: Using cryptographic hash for deterministic IDs
            person_id = stable_person_id(patient_source_id)
            
            gender_source = resource.get('gender', 'unknown')
            gender_concept_id = get_omop_gender_id(gender_source)
            
            birth_date = resource.get('birthDate')
            if birth_date:
                dt = datetime.strptime(birth_date, '%Y-%m-%d')
                yob, mob, dob = dt.year, dt.month, dt.day
            else:
                yob, mob, dob = 0, 0, 0
            
            return {
                'person_id': person_id,
                'gender_concept_id': gender_concept_id,
                'year_of_birth': yob,
                'month_of_birth': mob,
                'day_of_birth': dob,
                'person_source_value': patient_source_id,
                'gender_source_value': gender_source
            }
    return None

def load_person_to_duckdb(persons):
    """Creates the PERSON table and inserts the structured data."""
    print("🔌 Connecting to DuckDB...")
    
    try:
        with duckdb.connect(DB_PATH) as con:
            print("⏳ Creating PERSON table...")
            
            con.execute("""
                CREATE TABLE IF NOT EXISTS person (
                    person_id BIGINT PRIMARY KEY,
                    gender_concept_id INTEGER,
                    year_of_birth INTEGER,
                    month_of_birth INTEGER,
                    day_of_birth INTEGER,
                    person_source_value VARCHAR,
                    gender_source_value VARCHAR
                )
            """)
            
            print(f"🚀 Processing and inserting {len(persons)} patients...")
            
            for p in persons:
                con.execute("""
                    INSERT INTO person 
                    (person_id, gender_concept_id, year_of_birth, month_of_birth, day_of_birth, person_source_value, gender_source_value)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT (person_id) DO UPDATE SET
                        gender_concept_id = EXCLUDED.gender_concept_id,
                        year_of_birth = EXCLUDED.year_of_birth,
                        month_of_birth = EXCLUDED.month_of_birth,
                        day_of_birth = EXCLUDED.day_of_birth
                """, (
                    p['person_id'], p['gender_concept_id'], p['year_of_birth'], 
                    p['month_of_birth'], p['day_of_birth'], p['person_source_value'], 
                    p['gender_source_value']
                ))
            
            count = con.execute("SELECT COUNT(*) FROM person").fetchone()[0]
            print(f"✅ PERSON table updated! Total structured patients: {count}")
            
            print("\n🔎 Sample of formatted OMOP data (with stable IDs):")
            sample = con.execute("""
                SELECT person_id, person_source_value 
                FROM person LIMIT 3
            """).fetchall()
            
            for row in sample:
                print(f" - Stable OMOP ID: {row[0]:<12} | Original FHIR UUID: {row[1]}")
                
    except Exception as e:
        print(f"❌ Database error: {e}")

# EXECUTION BLOCK
print("⚙️ STARTING ETL PIPELINE (FHIR -> OMOP PERSON) [V2: STABLE IDs]\n" + "-"*50)
json_files = [f for f in os.listdir(FHIR_DIR) if f.endswith('.json')]
person_records = []

for file in json_files:
    record = extract_person_data(file)
    if record:
        person_records.append(record)

if person_records:
    load_person_to_duckdb(person_records)
else:
    print("❌ No valid FHIR patient files found.")

⚙️ STARTING ETL PIPELINE (FHIR -> OMOP PERSON) [V2: STABLE IDs]
--------------------------------------------------
🔌 Connecting to DuckDB...
⏳ Creating PERSON table...
🚀 Processing and inserting 58 patients...
✅ PERSON table updated! Total structured patients: 116

🔎 Sample of formatted OMOP data (with stable IDs):
 - Stable OMOP ID: 487152244    | Original FHIR UUID: 0190fb66-1ec0-482d-02cb-f08d3defbe04
 - Stable OMOP ID: 578478374    | Original FHIR UUID: 8299c680-3752-8234-12b1-e1be41316454
 - Stable OMOP ID: 631565143    | Original FHIR UUID: cd121f41-7f29-de03-090d-a6233ce7268b
